# Preprocessing et Feature Engineering

In [6]:
import pandas as pd
import numpy as np

## Date conversion and Data loading

In [7]:
df = pd.read_csv("../data/merged_dataset.csv")
df["delivery_time"] = pd.to_datetime(df['delivery_time'])

## Handling Missing Values

Linear interpolation for weather, as it's a continuous time series

In [8]:
df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)

Power is proportional to the cube of wind speed:

In [9]:
df['wind_speed_100m_cube'] = df['wind_speed_100m'] ** 3

## Feature Engineering: Wind Physics

Wind Shear: Difference in speed between 10m and 100m:

In [10]:
df['wind_shear'] = df['wind_speed_100m'] - df['wind_speed_10m']

## Feature Engineering: Circular Encoding (Direction & Time)

Wind Direction (avoiding the 359° to 0° jump)

In [11]:
df['wind_dir_100m_sin'] = np.sin(np.radians(df['wind_direction_100m']))
df['wind_dir_100m_cos'] = np.cos(np.radians(df['wind_direction_100m']))

Time-based features (Daily and Seasonal cycles)

In [12]:
df['hour'] = df['delivery_time'].dt.hour
df['month'] = df['delivery_time'].dt.month

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 23)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 23)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 11)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 11)

## Target Normalization (Critical for multi-site modeling)
Predicting Load Factor (0 to 1) instead of raw MW

In [13]:
# df['target'] = df['production'] / df['installed_capacity']

Remove columns that would cause data leakage or are redundant

In [14]:
cols_to_drop = [ 'hour', 'month', 'wind_direction_10m', 'wind_direction_100m'] # 'production',
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [15]:
df.head()

,delivery_time,production,installed_capacity,wind_speed_10m,wind_speed_100m,wind_gusts_10m,temperature_2m,dewpoint_2m,apparent_temperature,pressure_msl,...,latitude,longitude,wind_speed_100m_cube,wind_shear,wind_dir_100m_sin,wind_dir_100m_cos,hour_sin,hour_cos,month_sin,month_cos
0,2023-01-01 00:00:00+00:00,147.7025,171.0,14.603082,19.897738,20.7,12.25,8.85,4.282408,1005.6,...,51.66,2.8,7877.911982,5.294656,-0.633238,-0.773957,0.000000,1.000000,0.0,1.0
1,2023-01-01 01:00:00+00:00,146.1775,171.0,16.182089,21.681328,20.8,12.10,8.80,3.290131,1005.4,...,51.66,2.8,10191.958316,5.499239,-0.608820,-0.793309,0.269797,0.962917,0.0,1.0
2,2023-01-01 02:00:00+00:00,146.1800,171.0,17.969420,23.809662,24.1,11.85,9.50,2.291797,1006.4,...,51.66,2.8,13497.697496,5.840242,-0.751797,-0.659395,0.519584,0.854419,0.0,1.0
3,2023-01-01 03:00:00+00:00,146.5050,171.0,14.792228,19.860010,23.9,11.80,9.85,4.007824,1007.2,...,51.66,2.8,7833.185089,5.067782,-0.760323,-0.649545,0.730836,0.682553,0.0,1.0
4,2023-01-01 04:00:00+00:00,146.6950,171.0,15.001333,19.915070,19.7,11.75,9.30,3.694952,1007.9,...,51.66,2.8,7898.516174,4.913737,-0.753200,-0.657792,0.887885,0.460065,0.0,1.0


In [16]:
df.drop(columns=["Unnamed: 0"], inplace=True)

KeyError: "['Unnamed: 0'] not found in axis"

In [17]:
df.to_csv("../data/processed_dataset.csv")